In [48]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [49]:
df = pd.read_csv('spoofing-merged-gps-only.csv')


In [50]:
from sklearn.preprocessing import LabelEncoder
import sklearn as sk

X = df.drop(columns=['label', 'timestamp', 'time_utc_usec', 'timestamp_time_relative'])
y = df['label']

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

In [51]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = X.select_dtypes(include=['float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols)
    ]
)
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [52]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    'DT' : DecisionTreeClassifier(random_state=42),
    'RF' : RandomForestClassifier(random_state=42),
    'XGBoost' : XGBClassifier(eval_metric='mlogloss',random_state=42),
    'CatBoost' : CatBoostClassifier(verbose=0,random_seed=42),
    'LightGBM' : LGBMClassifier(verbose=-1,random_seed=42),
}

In [53]:
from sklearn.metrics import accuracy_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(name + ": " + str(accuracy_score(y_test, y_pred)))

DT: 1.0
RF: 1.0
XGBoost: 1.0
CatBoost: 1.0
LightGBM: 1.0


In [54]:
# Sprawdzenie, co zepsuło wynik (Feature Importance)
import matplotlib.pyplot as plt

xgb_model = models['XGBoost']
importances = pd.Series(xgb_model.feature_importances_, index=preprocessor.get_feature_names_out())

# Wyświetl 10 najbardziej "podejrzanych" kolumn
print(importances.sort_values(ascending=False).head(10))

num__lat_y                0.881579
num__hdop                 0.049968
num__lat_x                0.013525
num__alt_ellipsoid_x      0.013324
num__cog_rad              0.012847
num__c_variance_rad       0.009194
num__vel_m_s              0.008043
num__jamming_indicator    0.005552
num__q[0]                 0.003130
num__q[3]                 0.002720
dtype: float32
